# Параллельное преобразование XLSB в CSV

Положите блокнот рядом с папкой `download_reestr` и запустите ячейки по порядку. Вторая ячейка создаёт рядом файл `xlsb_parallel_runner.py`. Он нужен для запуска отдельных процессов из JupyterLab.

Первый запуск обрабатывает до **10 ещё не преобразованных файлов** и выводит время и грубую оценку на весь объём. Для полного запуска измените `TEST_FILES = None` и запустите последнюю ячейку снова. При `OVERWRITE = False` уже существующие CSV пропускаются. Если в папке остались неполные CSV от старой версии, смените `OUTPUT` на новую папку или включите `OVERWRITE = True`.

По умолчанию используются до четырёх процессов. Каждый процесс читает первый лист XLSB построчно и пишет CSV с запятыми в соответствующую подпапку `download_reestr_csv_misha`. Готовый CSV появляется после полной записи. Даты Excel могут выгружаться числовыми значениями.


In [ ]:
%pip install pyxlsb==1.0.10


In [ ]:
%%writefile xlsb_parallel_runner.py
import argparse
import csv
import os
import tempfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from time import perf_counter

from pyxlsb import open_workbook


def convert_file(source, target):
    started = perf_counter()
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    written = 0

    try:
        with open_workbook(str(source)) as workbook:
            with workbook.get_sheet(1) as sheet:
                with tempfile.NamedTemporaryFile(
                    mode="w", dir=target.parent, prefix=".xlsb-", suffix=".part",
                    encoding="utf-8-sig", newline="", delete=False,
                ) as output:
                    temporary = Path(output.name)
                    writer = csv.writer(output, delimiter=",")
                    next_row = 0
                    for row in sheet.rows(sparse=True):
                        if not row:
                            continue
                        row_number = row[0].r
                        if row_number > next_row:
                            blank_row = [""] * len(row)
                            for _ in range(next_row, row_number):
                                writer.writerow(blank_row)
                        # csv.writer сам превращает None в пустое поле.
                        writer.writerow([cell.v for cell in row])
                        next_row = row_number + 1
                    written = next_row
        temporary.replace(target)
        return written, perf_counter() - started
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--workers", type=int, required=True)
    parser.add_argument("--limit", type=int)
    parser.add_argument("--overwrite", action="store_true")
    options = parser.parse_args()
    if options.workers < 1 or (options.limit is not None and options.limit < 1):
        parser.error("Число процессов и лимит должны быть положительными.")
    if not options.source.is_dir():
        parser.error(f"Папка не найдена: {options.source.resolve()}")

    files = sorted(
        path for path in options.source.rglob("*")
        if path.is_file() and path.suffix.lower() == ".xlsb"
        and not path.name.startswith("~$")
    )
    pending = []
    skipped = 0
    destinations = set()
    for source in files:
        relative = source.relative_to(options.source)
        target = (options.output / relative).with_suffix(".csv")
        if target in destinations:
            parser.error(f"Несколько исходных файлов соответствуют одному CSV: {target}")
        destinations.add(target)
        if target.exists() and not options.overwrite:
            skipped += 1
        else:
            pending.append((source, target))

    # При пробном запуске берём файлы из разных частей списка.
    remaining = len(pending)
    if options.limit is not None and remaining > options.limit:
        pending = [pending[i * remaining // options.limit] for i in range(options.limit)]

    total = len(pending)
    workers = min(options.workers, os.cpu_count() or 1, total)
    print(f"Найдено: {len(files)} | пропущено готовых: {skipped} | в запуске: {total}", flush=True)
    if not total:
        return 0

    print(f"Параллельных процессов: {workers}", flush=True)
    started = perf_counter()
    converted = 0
    failures = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        jobs = {pool.submit(convert_file, source, target): source for source, target in pending}
        for done, job in enumerate(as_completed(jobs), start=1):
            relative = jobs[job].relative_to(options.source)
            try:
                rows, seconds = job.result()
                converted += 1
                status = f"{relative} | {rows:,} строк | {seconds:.1f} с"
            except Exception as error:
                failures.append(relative)
                status = f"{relative} | ОШИБКА: {type(error).__name__}: {error}"
            elapsed = perf_counter() - started
            filled = 20 * done // total
            bar = "#" * filled + "-" * (20 - filled)
            eta = elapsed / done * (total - done) / 60
            print(f"[{bar}] {done}/{total} | {status} | осталось ~{eta:.1f} мин", flush=True)

    elapsed = perf_counter() - started
    print(f"Готово: {converted}/{total} | ошибок: {len(failures)} | время: {elapsed / 60:.1f} мин", flush=True)
    if options.limit is not None and converted == total and remaining > total:
        estimate = elapsed * remaining / total / 60
        print(f"Грубая оценка для {remaining} файлов при той же скорости: {estimate:.0f} мин.", flush=True)
    return int(bool(failures))


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SOURCE = Path("download_reestr")
OUTPUT = Path("download_reestr_csv_misha")
WORKERS = min(4, os.cpu_count() or 1)
TEST_FILES = 10  # None — обработать все файлы.
OVERWRITE = False  # True — пересоздать существующие CSV.

command = [
    sys.executable, "-u", str(Path("xlsb_parallel_runner.py").resolve()),
    "--source", str(SOURCE.resolve()),
    "--output", str(OUTPUT.resolve()),
    "--workers", str(WORKERS),
]
if TEST_FILES is not None:
    command.extend(["--limit", str(TEST_FILES)])
if OVERWRITE:
    command.append("--overwrite")

env = dict(os.environ, PYTHONIOENCODING="utf-8")
with subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", env=env,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code:
    print(f"Код завершения: {return_code}. Подробности в выводе выше.")
